# Biomedical NLP: Disease & Chemical NER (BC5CDR, PubMedBERT)

Fine-tune **PubMedBERT-base** (a BERT model pretrained from scratch on PubMed abstracts) to find real
**disease** and **chemical** mentions inside raw biomedical text, token by token (BIO tagging), then use the
extracted entities to build a simple real chemical<->disease co-mention table.

~80-90% of the code below is left blank (`____`) for you to write from each cell's hint + the library docs.
Read `PROJECT_NARRATIVE.md` and `WORKFLOW.md` first (private, not in this repo's public version).

**Scoped deliberately short for a single Kaggle GPU session** (T4/P100): ~1500 total abstracts, 2-3 epochs,
small batch -- should train in well under an hour, comfortably inside Kaggle's 9-hour session cap.

## Step 0: Environment setup (GIVEN)
True boilerplate only -- installs + imports. Enable a GPU accelerator (Settings -> Accelerator -> GPU T4 x2
or P100) before running.

In [4]:
!pip install -q setuptools_scm
!pip install -q transformers datasets seqeval evaluate biopython accelerate

import os
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification, pipeline
)
import evaluate
from Bio import Entrez

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.2/123.2 kB 3.2 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.0 MB/s eta 0:00:0000:0100:01
device: cuda


## Step 1: Load the real BC5CDR dataset and describe it
**WHAT:** fetch the real, double-annotated BioCreative V CDR corpus -- 1500 real PubMed abstracts tagged for
disease and chemical mentions in BIO format.
**WHY (biology/NLP):** this is the real standard benchmark corpus for biomedical NER covering *both* entity
types this project needs (see WORKFLOW.md for why not NCBI Disease / scispaCy's own corpora).
**HOW:** `datasets.load_dataset`, dataset id `"tner/bc5cdr"`. Look up: `dataset["train"].features["tags"]`
(or `ner_tags` -- print the real feature names to check which key this loader actually uses) to get the
real BIO label list -- do NOT hardcode it, different loaders order labels differently.

In [18]:
import json
import requests
from datasets import Dataset, DatasetDict

# WHAT/WHY: tner/bc5cdr ships a Python *loading script* on the HF Hub, and the datasets library no
# longer supports script-based datasets at all (a real, external breaking change, not a local bug) --
# so we fetch the same real annotated JSON-lines files the script itself would have read, directly.
BASE_URL = "https://huggingface.co/datasets/tner/bc5cdr/resolve/main/dataset"
SPLIT_FILES = {"train": "train.json", "validation": "valid.json", "test": "test.json"}

def fetch_split(filename):
    resp = requests.get(f"{BASE_URL}/{filename}")
    resp.raise_for_status()
    records = [json.loads(line) for line in resp.text.strip().split("\n")]
    return Dataset.from_list(records)

raw_ds = DatasetDict({split: fetch_split(fname) for split, fname in SPLIT_FILES.items()})
print(raw_ds)

for split in raw_ds:
    print(split, len(raw_ds[split]))

# Real BIO label list for this specific loader -- confirmed directly against real fetched examples
# (e.g. "Famotidine" -> tag 1 = B-Chemical, "delirium" -> tag 2 = B-Disease, "renal failure" -> tags
# 2,3,3 = B-Disease + two I-Disease continuations, "fusidic acid" -> tags 1,4 = B-Chemical + I-Chemical).
# Not the usual alphabetical BIO order, so it had to be verified against real data, not assumed.
label_list = ["O", "B-Chemical", "B-Disease", "I-Disease", "I-Chemical"]
print("real label list:", label_list)

DatasetDict({
    train: Dataset({
        features: ['tags', 'tokens'],
        num_rows: 5228
    })
    validation: Dataset({
        features: ['tags', 'tokens'],
        num_rows: 5330
    })
    test: Dataset({
        features: ['tags', 'tokens'],
        num_rows: 5865
    })
})
train 5228
validation 5330
test 5865
real label list: ['O', 'B-Chemical', 'B-Disease', 'I-Disease', 'I-Chemical']


## Step 2: Tokenize + align word-level labels onto subword tokens
**WHAT:** PubMedBERT's tokenizer splits words into subword pieces; BC5CDR's real labels are per-word. Every
label must be re-mapped onto however many subword tokens its word became.
**WHY:** getting this wrong silently trains the model against scrambled labels -- no error, just a model that
never learns anything real (see WORKFLOW.md Section 6).
**HOW:** `AutoTokenizer.from_pretrained("microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract")`, then
`tokenizer(..., is_split_into_words=True)` + `tokenized.word_ids()` to find which original word each subword
token came from. Label the first subword of a word with its real label, every following subword of the same
word and all special/padding tokens with `-100` (the id PyTorch's loss function is told to ignore).

In [19]:
MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract"

# WRITE-IT: load the real PubMedBERT tokenizer
# hint: tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME) 
tokenizer.model_max_length = 512

def align_labels(example):
    # WRITE-IT: tokenize the already-split-into-words input, then re-align example's word-level label ids
    # (check the real column name holding the word list, e.g. "tokens") onto subword tokens using
    # tokenized.word_ids() -- first subword of a word keeps the real label, later subwords of the same word
    # and all special tokens get -100.
    # hint:
    #   tokenized = tokenizer(example["tokens"], truncation=True, is_split_into_words=True)
    #   word_ids = tokenized.word_ids()
    #   labels, prev_word = [], None
    #   for wid in word_ids:
    #       if wid is None:
    #           labels.append(-100)
    #       elif wid != prev_word:
    #           labels.append(example["tags"][wid])
    #       else:
    #           labels.append(-100)
    #       prev_word = wid
    #   tokenized["labels"] = labels
    #   return tokenized
    tokenized = tokenizer(example["tokens"], truncation=True, is_split_into_words = True)
    word_ids = tokenized.word_ids()
    labels, prev_word = [], None
    for wid in word_ids:
        if wid is None:
            labels.append(-100)
        elif wid != prev_word:
            labels.append(example["tags"][wid])
        else: 
            labels.append(-100)
        prev_word = wid
    tokenized["labels"] = labels
    return tokenized
# WRITE-IT: apply align_labels across every real split
# hint: tokenized_ds = raw_ds.map(align_labels, batched=False)
tokenized_ds = raw_ds.map(align_labels, batched=False)

print(tokenized_ds["train"][0])

Map:   0%|          | 0/5228 [00:00<?, ? examples/s]

Map:   0%|          | 0/5330 [00:00<?, ? examples/s]

Map:   0%|          | 0/5865 [00:00<?, ? examples/s]

{'tags': [1, 0, 0, 0, 0, 0, 1, 0], 'tokens': ['Naloxone', 'reverses', 'the', 'antihypertensive', 'effect', 'of', 'clonidine', '.'], 'input_ids': [2, 14007, 5967, 1026, 1680, 12237, 2224, 1685, 15753, 17, 3], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [-100, 1, 0, -100, 0, 0, 0, 0, 1, 0, -100]}


## Step 3: Load PubMedBERT with a token-classification head
**WHAT:** attach a fresh linear classification head (size = the real number of BIO labels) on top of
PubMedBERT's own pretrained token representations.
**HOW:** `AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=len(label_list))`. Also
build `id2label`/`label2id` dicts from the real `label_list` -- the Trainer and later the `pipeline()` demo
both need these to turn label ids back into real tag names.

In [20]:
# WRITE-IT: build id2label / label2id from the real label_list
# hint: id2label = {i: l for i, l in enumerate(label_list)}
#       label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}

# WRITE-IT: load the real PubMedBERT model with a token-classification head sized to the real label count
# hint: model = AutoModelForTokenClassification.from_pretrained(
#           MODEL_NAME, num_labels=len(label_list), id2label=id2label, label2id=label2id)
model = AutoModelForTokenClassification.from_pretrained(MODEL_NAME, num_labels=len(label_list), id2label=id2label, label2id=label2id)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

N

## Step 4: Entity-level evaluation with seqeval
**WHAT:** score whole predicted entities against real gold entities, not raw per-token accuracy (see
WORKFLOW.md Section 2 for why this is the metric that actually matters here).
**HOW:** `evaluate.load("seqeval")`. Inside `compute_metrics`, strip out every `-100` position (both from
predictions and real labels) before converting ids back to label strings and calling `seqeval.compute(...)`.

In [21]:
seqeval_metric = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    # WRITE-IT: convert raw logits -> predicted label ids (argmax), drop every position where the real
    # label is -100 (both from predictions and true labels), convert surviving ids to label strings via
    # id2label, then call seqeval_metric.compute(predictions=..., references=...) and return its real
    # overall precision/recall/f1/accuracy (seqeval's output dict has "overall_precision" etc. keys).
    # hint:
    #   logits, labels = eval_pred
    #   preds = np.argmax(logits, axis=2)
    #   true_labels, true_preds = [], []
    #   for pred_row, label_row in zip(preds, labels):
    #       cur_p, cur_l = [], []
    #       for p, l in zip(pred_row, label_row):
    #           if l != -100:
    #               cur_p.append(id2label[p]); cur_l.append(id2label[l])
    #       true_preds.append(cur_p); true_labels.append(cur_l)
    #   results = seqeval_metric.compute(predictions=true_preds, references=true_labels)
    #   return {"precision": results["overall_precision"], "recall": results["overall_recall"],
    #           "f1": results["overall_f1"], "accuracy": results["overall_accuracy"]}
    logits, labels = eval_pred 
    preds = np.argmax(logits, axis = 2)
    true_labels, true_preds = [], []
    for pred_row, label_row in zip(preds, labels):
        cur_p, cur_l = [], []
        for p, l in zip(pred_row, label_row):
            if l != -100: 
                cur_p.append(id2label[p]); cur_l.append(id2label[l])
        true_preds.append(cur_p); true_labels.append(cur_l)
    results = seqeval_metric.compute(predictions=true_preds, references=true_labels) 
    return {"precision": results["overall_precision"], "recall": results["overall_recall"], "f1":results["overall_f1"], "accuracy":results["overall_accuracy"]}

## Step 5: Train (short, Kaggle-session-scoped) + evaluate on the real test set
**WHAT:** fine-tune for 2-3 epochs, small batch, fp16 -- deliberately short (see PROJECT_NARRATIVE.md Section
5 for why), then report the real held-out precision/recall/F1.
**HOW:** `TrainingArguments(..., num_train_epochs=3, per_device_train_batch_size=16, fp16=True,
eval_strategy="epoch")`, `DataCollatorForTokenClassification(tokenizer)`, `Trainer(...)`, `trainer.train()`,
then `trainer.evaluate(tokenized_ds["test"])`.

In [22]:
# WRITE-IT: build TrainingArguments scoped for a short run (2-3 epochs is enough -- see WORKFLOW.md)
# hint: args = TrainingArguments(
#           output_dir="ner_ckpt", num_train_epochs=3, per_device_train_batch_size=16,
#           per_device_eval_batch_size=16, eval_strategy="epoch", save_strategy="no",
#           fp16=torch.cuda.is_available(), logging_steps=50, report_to="none")
args = TrainingArguments(output_dir="ner_ckpt", num_train_epochs=12, per_device_train_batch_size=16, per_device_eval_batch_size=16, eval_strategy="epoch", save_strategy="no", fp16=torch.cuda.is_available(), logging_steps=50, report_to="none")

# WRITE-IT: a real data collator for dynamic padding of token-classification batches
# hint: collator = DataCollatorForTokenClassification(tokenizer)
collator = DataCollatorForTokenClassification(tokenizer)

# WRITE-IT: build the Trainer -- pick a real validation split name from tokenized_ds (print tokenized_ds to
# check: it may be "validation" or "valid")
# hint: trainer = Trainer(model=model, args=args, train_dataset=tokenized_ds["train"],
#           eval_dataset=tokenized_ds["validation"], data_collator=collator,
#           compute_metrics=compute_metrics)
trainer = Trainer(model=model, args=args, train_dataset=tokenized_ds["train"],
                  eval_dataset=tokenized_ds["validation"], data_collator = collator, compute_metrics=compute_metrics)

trainer.train()

/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precison,Recall,F1,Accuracy
1,0.135339,0.136050,0.886800,0.914816,0.900590,0.978111
2,0.061391,0.154193,0.891635,0.899072,0.895338,0.979359
3,0.034182,0.176127,0.870376,0.919925,0.894465,0.978175
4,0.017980,0.196163,0.902404,0.912001,0.907177,0.980681
5,0.009482,0.251799,0.903983,0.910958,0.907457,0.979019
6,0.004766,0.239253,0.891148,0.918465,0.904601,0.980057
7,0.002187,0.253788,0.903018,0.920342,0.911598,0.980690
8,0.002950,0.265598,0.902033,0.920655,0.911249,0.980580
9,0.000806,0.270978,0.906498,0.917840,0.912133,0.981103
10,0.000869,0.272839,0.906022,0.917735,0.911841,0.981204


TrainOutput(global_step=1968, training_loss=0.03186627636106147, metrics={'train_runtime': 867.9258, 'train_samples_per_second': 72.283, 'train_steps_per_second': 2.267, 'total_flos': 2474876072405880.0, 'train_loss': 0.03186627636106147, 'epoch': 12.0})

In [23]:
# WRITE-IT: evaluate on the real held-out TEST split (not validation) and print the real numbers
# hint: test_metrics = trainer.evaluate(tokenized_ds["test"])
#       print(test_metrics)
test_metrics = trainer.evaluate(tokenized_ds["test"])
print(test_metrics)

pd.DataFrame([test_metrics]).to_csv(os.path.join(RESULTS_DIR, "test_metrics.csv"), index=False)

Training Loss,Validation Loss,Epoch,Precison,Recall,F1,Accuracy
0.000564,0.305996,12,0.884547,0.918544,0.901225,0.979324


{'eval_loss': 0.3059964179992676, 'eval_precison': 0.8845474180247398, 'eval_recall': 0.9185441941074524, 'eval_f1': 0.9012253063265816, 'eval_accuracy': 0.9793239223507969}


## Step 6: Live demo on real, unseen PubMed abstracts
**WHAT:** fetch a handful of real, current PubMed abstracts via Entrez (never seen during training) and run
the fine-tuned model on them -- a direct, honest check of generalization beyond BC5CDR (see
PROJECT_NARRATIVE.md Section 2/6).
**HOW:** `Entrez.email = "<your real email>"`, `Entrez.esearch`/`Entrez.efetch` for a handful of real PMIDs on
a chemical/disease-relevant query (e.g. a real drug name), then `transformers.pipeline("ner", model=model,
tokenizer=tokenizer, aggregation_strategy="simple")` on the fetched abstract text.

In [24]:
Entrez.email = "yashraj.7156@gmail.com"  # your real email -- NCBI requires this for E-utilities

# WRITE-IT: search PubMed for a handful of real, recent abstracts on a chemical/disease-relevant query
# hint:
#   handle = Entrez.esearch(db="pubmed", term="metformin AND nephropathy", retmax=5, sort="most recent")
#   record = Entrez.read(handle)
#   pmids = record["IdList"]
handle= Entrez.esearch(db="pubmed", term="metformin AND nephropathy", retmax=5, sort="most recent")
record = Entrez.read(handle) 
pmids = record["IdList"]
print("real PMIDs fetched:", pmids)

# WRITE-IT: fetch the real abstract text for those PMIDs
# hint:
#   handle = Entrez.efetch(db="pubmed", id=pmids, rettype="abstract", retmode="text")
#   abstracts_text = handle.read()
handle = Entrez.efetch(db= "pubmed", id = pmids, rettype="abstract", retmode="text")
abstracts_text = handle.read()
print(abstracts_text[:500])

real PMIDs fetched: ['42883700', '42857026', '42835393', '42816183', '42815506']
1. Int J Mol Sci. 2026 Sep 22;27(19):8451. doi: 10.3390/ijms27198451.

Mitochondrial Dynamics in Diabetes Mellitus and Its Complications: Mechanisms, 
Pathological Roles, and Therapeutic Implications.

Wang G(1), Zhao Y(1), Liu G(1), Du Y(1), Liu S(1), Yang Y(1), Bu S(1), Zhao 
B(2).

Author information:
(1)Department of Pharmacology of Chinese Materia Medica, School of Chinese 
Materia Medica, Beijing University of Chinese Medicine, Beijing 102488, China.
(2)Beijing Research Institute of Chines


In [25]:
# WRITE-IT: build a HF pipeline with the fine-tuned model + tokenizer, then run it on the real fetched text
# hint: ner_pipe = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple",
#           device=0 if torch.cuda.is_available() else -1)
#       live_entities = ner_pipe(abstracts_text)
ner_pipe = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple",
           device=0 if torch.cuda.is_available() else -1)
live_entities = ner_pipe(abstracts_text)

pd.DataFrame(live_entities).to_csv(os.path.join(RESULTS_DIR, "live_demo_entities.csv"), index=False)
print(pd.DataFrame(live_entities).head(20))

   entity_group     score                       word  start   end
0       Disease  0.999904          diabetes mellitus     97   114
1       Disease  0.999909          diabetes mellitus    577   594
2       Disease  0.999896                         dm    596   598
3       Disease  0.995040         metabolic disorder    617   635
4       Disease  0.999883  mitochondrial dysfunction    646   671
5       Disease  0.999857     diabetic complications    756   778
6       Disease  0.963601                dysfunction   1306  1317
7       Disease  0.992297                    insulin   1319  1326
8       Disease  0.999916       diabetic nephropathy   1339  1360
9       Disease  0.999814                         dn   1362  1364
10      Disease  0.999916    diabetic cardiomyopathy   1367  1390
11      Disease  0.999782                        dcm   1392  1395
12      Disease  0.999914       diabetic retinopathy   1398  1418
13      Disease  0.999489                         dr   1420  1422
14      Di

## Step 7: Real chemical<->disease co-mention table + final exports
**WHAT:** using the real, gold-labeled TEST set entities (not the model's own live predictions -- see
PROJECT_NARRATIVE.md Section 5 for why), count how often a chemical and a disease are mentioned in the same
real abstract -- a simple, concrete first step toward literature-scale relationship mining.
**HOW:** reconstruct gold entity spans per test abstract from `tokenized_ds["test"]`'s real tokens + tags,
then build a same-document co-occurrence count table with `pandas` (e.g. `groupby`/`pivot_table` or a plain
nested loop over each abstract's distinct chemical/disease entity sets).

In [26]:
# WRITE-IT: for each real test-set example, extract the set of distinct gold disease entity strings and
# the set of distinct gold chemical entity strings (reconstruct multi-token entities from consecutive
# B-/I- tags using the real label_list), then count co-occurrences of each (chemical, disease) pair across
# abstracts into a real pandas DataFrame with columns ["chemical", "disease", "co_mention_count"].
# hint: write a small helper that walks a list of (token, tag_name) pairs and yields complete entity strings
# whenever a B-tag starts (joining consecutive I-tags of the same type), call it once per test example for
# each entity type, then use collections.Counter over all (chemical, disease) pairs found in the same
# abstract, and build the final DataFrame from Counter.most_common().
from collections import Counter

def extract_entities(tokens, tags, label_list, entity_type):
    """
    Walk one abstract's (word, tag_id) sequence and pull out every complete entity
    string of one type ("Disease" or "Chemical"), joining consecutive I-tags onto
    the B-tag that started them.
    """
    entities = []
    current_words = []

    for token, tag_id in zip(tokens, tags):
        tag_name = label_list[tag_id]          # e.g. "B-Disease", "I-Chemical", "O"

        if tag_name == f"B-{entity_type}":
            # a new entity of this type starts -- flush whatever was being built, start fresh
            if current_words:
                entities.append(" ".join(current_words))
            current_words = [token]

        elif tag_name == f"I-{entity_type}" and current_words:
            # continuation of the entity currently being built
            current_words.append(token)

        else:
            # "O", or a tag belonging to the OTHER entity type -- close out what we had
            if current_words:
                entities.append(" ".join(current_words))
            current_words = []

    if current_words:                           # don't drop an entity that runs to the end
        entities.append(" ".join(current_words))

    return entities

# WRITE-IT: for each real test-set example, extract the set of distinct gold disease entity strings and
# the set of distinct gold chemical entity strings, then count co-occurrences of each (chemical, disease)
# pair across abstracts into a real pandas DataFrame.
pair_counts = Counter()

for example in raw_ds["test"]:
    tokens = example["tokens"]
    tags = example["tags"]

    diseases = set(extract_entities(tokens, tags, label_list, "Disease"))
    chemicals = set(extract_entities(tokens, tags, label_list, "Chemical"))

    for chem in chemicals:
        for dis in diseases:
            pair_counts[(chem, dis)] += 1       # same abstract -> co-mention

comention_df = pd.DataFrame(
    [(chem, dis, count) for (chem, dis), count in pair_counts.most_common()],
    columns=["chemical", "disease", "co_mention_count"]
)

comention_df.to_csv(os.path.join(RESULTS_DIR, "chemical_disease_comentions.csv"), index=False)
print(comention_df.sort_values("co_mention_count", ascending=False).head(15))

print("real results exported to", RESULTS_DIR)

           chemical              disease  co_mention_count
0       pilocarpine             seizures                18
1       pilocarpine   status epilepticus                15
2       haloperidol            catalepsy                14
3           cocaine           chest pain                14
4       doxorubicin       cardiotoxicity                13
5           heparin     thrombocytopenia                10
6       pilocarpine              seizure                10
7          levodopa                   PD                 9
8          levodopa           dyskinesia                 8
9               DOX       cardiotoxicity                 8
10         levodopa  Parkinson's disease                 7
11      doxorubicin             toxicity                 7
12          cocaine             seizures                 7
13          cocaine                   MI                 7
14  methamphetamine            psychosis                 6
real results exported to results
